In [0]:
# Databricks notebook source
# Data cleansing

spark.sql("DECLARE OR REPLACE VARIABLE filas_antes BIGINT")
spark.sql("SET VAR filas_antes = (SELECT count(*) FROM workspace.default.silver_ventas)")

# COMMAND ----------

spark.sql("""
CREATE OR REPLACE TABLE workspace.default.silver_ventas
CLUSTER BY (store, date)
AS
WITH t AS (
  SELECT s.* EXCEPT (quantity, _source_file, _n_cols,
                     menu_price, total, item_discount, total_discount, tax,
                     item_voided, order_voided),
         try_cast(s.menu_price AS DECIMAL(12,4)) AS menu_price,
         try_cast(s.total AS DECIMAL(12,4)) AS total,
         try_cast(s.item_discount AS DECIMAL(12,4)) AS item_discount,
         try_cast(s.total_discount AS DECIMAL(12,4)) AS total_discount,
         try_cast(s.tax AS DECIMAL(12,4)) AS tax,
         try_cast(s.item_voided AS DECIMAL(12,4)) AS item_voided,
         try_cast(s.order_voided AS DECIMAL(12,4)) AS order_voided,
         timestampdiff(MINUTE, s.begin_time, s.end_time) AS duracion_min
  FROM workspace.default.silver_ventas s
)
SELECT store,
       pos_code,
       check_number,
       menu_item,
       transaction_type,
       -- Reducir categorías de tender_type: el UPDATE de Genie y su arreglo con spark.sql() en un solo CASE
       --   NULL → NULL (sin cambio)
       --   Billetes puros o combinaciones de solo billetes → Cash
       --   Combinaciones con tarjeta (cash+tarjeta, dos tarjetas, billete+tarjeta) → Hybrid
       --   Tarjetas individuales → se quedan como están
       -- El valor vacío va en la lista porque así corrió el UPDATE original.
       CASE
         -- Billetes individuales → Cash
         WHEN tender_type IN ('Cash', '', '$1', '$5', '$10', '$20', '$50')
         THEN 'Cash'
         -- Compuesto con al menos una tarjeta/plataforma → Hybrid
         WHEN contains(tender_type, ':')
              AND (
                tender_type LIKE '%Visa%'
                OR tender_type LIKE '%MasterCard%'
                OR tender_type LIKE '%Discover%'
                OR tender_type LIKE '%American Express%'
                OR tender_type LIKE '%OLO%'
                OR tender_type LIKE '%EZCater%'
                OR tender_type LIKE '%FB DoorDash%'
                OR tender_type LIKE '%Online Orders%'
                OR tender_type LIKE '%EOD Tender%'
              )
         THEN 'Hybrid'
         -- Compuesto de solo billetes → Cash
         WHEN contains(tender_type, ':')
         THEN 'Cash'
         -- Tarjetas individuales → sin cambio
         ELSE tender_type
       END AS tender_type,
       report_group,
       CASE pop
         WHEN 'To Go'  THEN 'Drive Thru'
         WHEN 'Inside' THEN 'Dine-In'
         WHEN '3'      THEN 'Carry Out'
         ELSE pop
       END AS pop,
       date,
       begin_time,
       end_time,
       menu_price,
       total,
       item_discount,
       total_discount,
       tax,
       item_voided,
       order_voided,
       duracion_min
FROM t
WHERE store IS NOT NULL
  AND pos_code IS NOT NULL
  AND check_number IS NOT NULL
  AND menu_price IS NOT NULL
  AND total IS NOT NULL
  AND item_discount IS NOT NULL
  AND total_discount IS NOT NULL
  AND tax IS NOT NULL
  AND item_voided IS NOT NULL
  AND order_voided IS NOT NULL
  AND transaction_type IS NOT NULL
  AND pop IS NOT NULL
  AND date IS NOT NULL
  AND begin_time IS NOT NULL
  AND end_time IS NOT NULL
  AND duracion_min IS NOT NULL
""")

# COMMAND ----------

spark.sql("""
SELECT
  filas_antes,
  count(*)               AS filas_despues,
  filas_antes - count(*) AS filas_quitadas
FROM workspace.default.silver_ventas
""").show()